In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df_header = pd.read_csv('../data/raw/accepted_2007_to_2018Q4.csv',nrows=0)

In [4]:
print(f"Total columns: {len(df_header.columns)}")

Total columns: 151


In [5]:
print(df_header.columns.tolist())

['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq',

In [6]:
loan_status_counts = pd.read_csv("../data/raw/accepted_2007_to_2018Q4.csv", usecols=['loan_status'])

In [7]:
print(loan_status_counts['loan_status'].value_counts())

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64


In [8]:
target_statuses = ['Fully Paid', 'Charged Off', 'Default']

df_status = pd.read_csv(
    '../data/raw/accepted_2007_to_2018Q4.csv',
    usecols=['loan_status']
)

mask = df_status['loan_status'].isin(target_statuses)
print(f"Rows kept: {mask.sum()}")
print(f"Rows dropped: {(~mask).sum()}")

Rows kept: 1345350
Rows dropped: 915351


In [9]:
safe_features = [
    'loan_amnt', 'term', 'installment', 'emp_length', 'home_ownership',
    'annual_inc', 'verification_status', 'purpose', 'addr_state', 'dti',
    'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths',
    'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'initial_list_status',
    'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal',
    'mort_acc', 'pub_rec_bankruptcies', 'tax_liens', 'mo_sin_old_rev_tl_op',
    'num_actv_bc_tl', 'num_actv_rev_tl', 'num_tl_op_past_12m',
    'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'avg_cur_bal', 'bc_open_to_buy',
    'bc_util', 'total_bc_limit', 'total_il_high_credit_limit'
]

cols_to_load = ['loan_status'] + safe_features

In [10]:
df = pd.read_csv('../data/raw/accepted_2007_to_2018Q4.csv', usecols = cols_to_load)

In [11]:
df = df[df['loan_status'].isin(target_statuses)].copy()

In [12]:
df['default'] = df['loan_status'].apply(lambda x: 0 if x == 'Fully Paid' else 1)

In [13]:
print(df.shape)
print(df['default'].value_counts(normalize = True))

(1345350, 42)
default
0    0.80035
1    0.19965
Name: proportion, dtype: float64


In [14]:
df.info()

<class 'pandas.DataFrame'>
Index: 1345350 entries, 0 to 2260697
Data columns (total 42 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   loan_amnt                   1345350 non-null  float64
 1   term                        1345350 non-null  str    
 2   installment                 1345350 non-null  float64
 3   emp_length                  1266834 non-null  str    
 4   home_ownership              1345350 non-null  str    
 5   annual_inc                  1345350 non-null  float64
 6   verification_status         1345350 non-null  str    
 7   loan_status                 1345350 non-null  str    
 8   purpose                     1345350 non-null  str    
 9   addr_state                  1345350 non-null  str    
 10  dti                         1344976 non-null  float64
 11  earliest_cr_line            1345350 non-null  str    
 12  fico_range_low              1345350 non-null  float64
 13  fico_range_hi

In [15]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing/len(df)*100).round(2)

In [16]:
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count']>0]
print(missing_summary)

                            missing_count  missing_pct
mths_since_last_record            1116786        83.01
mths_since_last_delinq             678761        50.45
emp_length                          78516         5.84
pct_tl_nvr_dlq                      67681         5.03
avg_cur_bal                         67549         5.02
mo_sin_old_rev_tl_op                67528         5.02
num_actv_rev_tl                     67527         5.02
tot_coll_amt                        67527         5.02
num_actv_bc_tl                      67527         5.02
tot_cur_bal                         67527         5.02
total_il_high_credit_limit          67527         5.02
num_tl_op_past_12m                  67527         5.02
bc_util                             61914         4.60
percent_bc_gt_75                    61557         4.58
bc_open_to_buy                      61145         4.54
mort_acc                            47281         3.51
total_bc_limit                      47281         3.51
revol_util

In [17]:
df.to_csv('../data/processed/loan_data_filtered.csv', index=False)
print(f"Saved {df.shape[0]} rows, {df.shape[1]} columns to data/processed/loan_data_filtered.csv")

Saved 1345350 rows, 42 columns to data/processed/loan_data_filtered.csv


In [3]:
df = pd.read_csv('../data/processed/loan_data_filtered.csv')
print(df.shape)

(1345350, 42)
